<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>

## Systems Thinking · From a Real Problem to an Optimized Decision

**Systems thinking turns a vague real-world problem into a simplified system that can be evaluated and improved.**

The lecture follows one continuous journey:

> **1. Frame the problem → 2. Select elements → 3. Build the system equation → 4. Simulate behavior → 5. Evaluate performance → 6. Select a preferred feasible decision**

**1 · Frame the real problem**

“The room is uncomfortable” is a symptom. A system model begins by defining the purpose and boundary: keep one classroom comfortable over the period of interest while accounting for energy use. A solution is not chosen yet.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/01-1_systems_thinking/assets/01_real_problem_labeled.png?v=71d35fa" alt="Labeled classroom problem showing discomfort, weather, cooling, and indoor temperature" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The picture separates the observed symptom from possible influences: weather, occupants, openings, cooling, indoor temperature, energy, and time.

**2 · Select the elements that matter**

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/01-1_systems_thinking/assets/02_candidate_elements_labeled.png?v=71d35fa" alt="Labeled candidate classroom elements including temperatures, occupants, openings, cooling, energy, and time" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The icons are candidates, not yet a system. The simplified model keeps outdoor temperature, occupants, indoor temperature, cooling, energy, and time. Door and window effects are omitted in this first model. This is a modeling choice: include what is needed for the decision, then revise the boundary if the model is insufficient.


**3 · Assign roles and build the system equation**

Each selected element receives a role. The role determines whether the element enters the system as a given value, an uncontrolled input, an evolving state, a decision, an output, a limit, or an analyst setting.

| Role | Question answered |
|:---|:---|
| Fixed parameter | What is treated as given? |
| External input | What changes but is not controlled? |
| System state | What condition carries forward over time? |
| Decision variable | What action can be chosen? |
| Performance output | What result is measured? |
| Constraint | What must every acceptable candidate satisfy? |
| Hyperparameter | What analyst setting changes evaluation or search? |

Changing is not the same as choosing. A decision acts on the real system; a hyperparameter acts on the analysis.

The roles define three connected mappings:

<div style="height: 0.4rem;"></div>

> $\displaystyle \text{Next state}=F(\text{current state},\text{external input},\text{decision};\text{fixed parameters})$
>
> $\displaystyle \text{Performance}=G(\text{state path},\text{decision};\text{fixed parameters})$
>
> $\displaystyle \text{Score}=H(\text{performance};\text{hyperparameter})$

<div style="height: 0.65rem;"></div>

\(F\) describes system behavior, \(G\) measures the resulting performance, and \(H\) evaluates that performance. The semicolon separates values treated as fixed during one analysis.

The classroom model now has a clear structure:

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/01-1_systems_thinking/assets/03_simplified_system_labeled.png?v=71d35fa" alt="Labeled system diagram mapping outdoor temperature, occupancy, cooling decision, fixed effects, and indoor state variables" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Outdoor temperature and occupants add heat, while cooling removes heat. This relationship becomes the state equation:

<div style="height: 0.4rem;"></div>

> $\displaystyle T_{t+1}=T_t+a\left(T_t^{\mathrm{out}}-T_t\right)+bN_t-cu_t$

<div style="height: 0.65rem;"></div>

| Variable | Meaning | Role |
|:---|:---|:---|
| \(T_t\) | Current indoor temperature | System state |
| \(T_t^{\mathrm{out}}\) | Outdoor temperature | External input |
| \(N_t\) | Number of occupants | External input |
| \(u_t\) | Cooling level | Decision variable |
| \(a,b,c\) | Weather, occupant-heat, and cooling effects | Fixed parameters |

At each time step, the model starts with \(T_t\), adds weather and occupant effects, subtracts the cooling effect, and produces \(T_{t+1}\). Repeating this update produces the complete temperature path.

The energy weight does not appear in the state equation. It cannot change the temperature path for a fixed cooling decision; it is introduced later when alternatives are compared.


In [ ]:
# Load the attached demo code in VS Code/Jupyter or JSPCV Playground.
from pathlib import Path
import importlib.util
import numpy
import matplotlib

_demo_paths = (
    Path("concept_demo.py"),
    Path("01-1_systems_thinking/concept_demo.py"),
    Path("uploads/concept_demo.py"),
)
_demo_path = next((path for path in _demo_paths if path.exists()), None)
if _demo_path is None:
    raise FileNotFoundError("Upload concept_demo.py together with this notebook.")

_spec = importlib.util.spec_from_file_location("concept_demo", _demo_path)
concept_demo = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(concept_demo)

_expected_demo_version = "2026-09-01-controlled-comparison-v2"
if getattr(concept_demo, "DEMO_VERSION", None) != _expected_demo_version:
    raise RuntimeError("Replace the attached concept_demo.py with the current version.")

concept_demo.show_system_behavior()


**4 · Simulate the system and visualize its behavior**

The code above compares three cooling decisions over 12 time steps. Outdoor temperature remains at 31 °C and occupancy remains at 20 people for every candidate. Only the cooling decision changes, so differences in the result can be attributed to that decision.

- The **left graph** shows low, moderate, and strong cooling schedules under the same external conditions.
- The **right graph** shows the temperature path produced by each schedule. Low cooling leaves the room hot, moderate cooling moves it into the comfort range, and strong cooling eventually makes it too cold.

The paths separate even though outdoor temperature and occupancy are identical. The model therefore makes the causal chain visible: **decision → system behavior → performance**.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/01-1_systems_thinking/assets/04_system_performance_labeled.png?v=71d35fa" alt="Labeled candidate-performance diagram connecting decision, state path, discomfort, energy, and feasibility" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

**5 · Measure performance and test feasibility**

The temperature path and cooling schedule are converted into two raw performance measures:

<div style="height: 0.4rem;"></div>

> $\displaystyle D=\operatorname{Discomfort}(T_1,\ldots,T_n)$
>
> $\displaystyle E=\operatorname{Energy}(u_1,\ldots,u_n)$

<div style="height: 0.65rem;"></div>

Discomfort \(D\) increases when the temperature leaves the 22–24 °C comfort range. Energy \(E\) increases with cooling effort. These values describe the candidate before any analyst preference is applied.

A candidate is feasible only when every requirement holds:

| Requirement | Allowed range |
|:---|:---|
| Early and late cooling | 0 to 5 |
| Indoor temperature | 20 °C to 30 °C |
| Total energy | At most 60 |

The evaluation sequence for one candidate is therefore:

> **Choose \(u\) → simulate \(T\) → calculate \(D\) and \(E\) → check the constraints**

An infeasible candidate is rejected, regardless of its score.

**6 · Compare feasible candidates and select a preferred decision**

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/01-1_systems_thinking/assets/05_system_optimization_wide_labeled.png?v=71d35fa" alt="Labeled optimization diagram connecting decision, system state, raw performance, energy weight, score, and preferred candidate" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The objective combines discomfort and energy for each feasible decision:

<div style="height: 0.4rem;"></div>

> $\displaystyle J(u;\lambda_E)=D(u)+\lambda_EE(u)$
>
> $\displaystyle \text{Preferred decision}=\text{feasible }u\text{ with the lowest }J$

<div style="height: 0.65rem;"></div>

Here, \(u\) is the decision and \(\lambda_E\) is the energy-weight hyperparameter. Changing \(u\) changes the physical system; changing \(\lambda_E\) changes only how the same performance is scored.

The optimization follows a transparent search:

1. Generate candidate pairs of early and late cooling levels on a 0.5-unit grid.
2. Simulate the temperature path for each pair.
3. Calculate \(D\) and \(E\), then reject infeasible candidates.
4. Calculate \(J\) for every feasible candidate.
5. Select the feasible grid candidate with the lowest \(J\).

The grid keeps the search visible. Its star marks the best sampled candidate, not a guaranteed optimum over every possible continuous value.

The interactive figure makes the complete chain visible:

- **Decision change:** the temperature path in Panel 1 and the square in Panels 2–3 move together. Changes in \(D\), \(E\), and feasibility show the physical consequence of that decision.
- **Current candidate versus search result:** the square represents the slider-selected candidate; the star represents the lowest-scoring feasible grid candidate. When they do not coincide, the current choice is not the grid benchmark.
- **Hyperparameter change with the decision fixed:** the temperature path, square, \(D\), and \(E\) stay unchanged. The score landscape and star can change because \(\lambda_E\) changes the comparison rule, not the physical system.

The result is not merely a final number. It is a traceable journey from a real problem, through a system model and measurable performance, to a preferred feasible decision under a stated evaluation rule.


In [ ]:
explorer = concept_demo.show_interactive_explorer()
systems_thinking_state = explorer.state
